In [ ]:
%%capture
%pip install manim
%pip install numpy

In [ ]:
from manim import *
import numpy as np
from manim import logger
import logging
logger.setLevel(logging.WARNING)

In [ ]:
# Cell 1: Setup (run once)
# ---------- Helpers ----------
def make_title(scene: Scene, text="Similarity Preservation in Spectral Hashing"):
    title = Text(text, font_size=40)
    scene.play(Write(title))
    scene.play(title.animate.to_edge(UP))
    return title


# ---------- Data ----------


In [ ]:
%%manim -qh -v WARNING Similarity_Title_And_Spectra
# Cell 2: Title + Original Spectra
class Similarity_Title_And_Spectra(Scene):
    def construct(self):
        title = make_title(self)

        spec_title = Text("Step 1: Original Spectra", font_size=28).next_to(title, DOWN, buff=0.5)
        self.play(Write(spec_title))

        spectra_group = VGroup()
        for i, label in enumerate(["s_1", "s_2", "s_3", "s_4"]):
            box = Rectangle(width=1.5, height=0.8, color=BLUE)
            txt = MathTex(label, font_size=30)
            spectra_group.add(VGroup(box, txt).arrange(DOWN, buff=0.1))

        spectra_group.arrange(RIGHT, buff=0.5).shift(DOWN * 0.5)
        self.play(Create(spectra_group))
        self.play(FadeOut(spec_title,spectra_group))

In [ ]:
%%manim -qh -v WARNING Similarity_Feature_Vectors
from manim import *
# Cell 2: hashed and non-hashed feature vectors introduction
class Similarity_Feature_Vectors(Scene):
    def construct(self):
        title = make_title(self)
        t = Text("Step 2: Non-hashed Feature Vectors", font_size=28).next_to(title, DOWN, buff=0.5)
        self.play(Write(t))

        non_hashed_group = VGroup()
        for i, label in enumerate(["f_1", "f_2", "f_3", "f_4"]):
            box = Rectangle(width=2, height=0.6, color=GREEN)
            lbl = MathTex(label, font_size=25)
            dim = Text("(8800-dim)", font_size=16, color=GRAY)
            non_hashed_group.add(VGroup(box, lbl, dim).arrange(DOWN, buff=0.05))

        non_hashed_group.arrange(RIGHT, buff=0.3).shift(UP * 0.5)
        self.play(Create(non_hashed_group))
        self.play(FadeOut(t))
        t = Text("Step 3: Hashed Feature Vectors", font_size=28).next_to(title, DOWN, buff=0.5)
        self.play(Write(t))

        hashed_group = VGroup()
        for i, label in enumerate(["t_1", "t_2", "t_3", "t_4"]):
            box = Rectangle(width=0.5, height=0.3, color=RED)
            lbl = MathTex(label, font_size=20)
            dim = Text("(400-dim)", font_size=16, color=GRAY)
            hashed_group.add(VGroup(box, lbl, dim).arrange(DOWN, buff=0.05))
        hashed_group.arrange(RIGHT, buff=0.3).shift(DOWN * 1.5)
        
        arrows = VGroup()
        for i in range(4):
            arrow = Arrow(
            non_hashed_group[i].get_bottom(),
            hashed_group[i].get_top(),
            color=YELLOW,
            stroke_width=2
            )
            arrows.add(arrow)

        self.play(Create(arrows))
        self.play(Create(hashed_group))

        self.wait(0.5)

In [ ]:
%%manim -qh -v WARNING Similarity_Pairwise_Explanation

def create_empty_matrix_with_subscript_labels(position, prefix, color):
    matrix_group = VGroup()
    if prefix == "f":
        prefix_color = GREEN
    else:
        prefix_color = RED
    # matrix cells
    matrix = VGroup()
    for i in range(4):
        row = VGroup()
        for j in range(4):
            is_diag = (i == j)
            
            cell = Rectangle(width=1.2, height=1.0, color=WHITE, fill_opacity=0.3, stroke_color=WHITE)
            if is_diag:
                text = MathTex(r"1", font_size=27, color=BLACK)
            else:
                text = MathTex(f"cs({prefix}_{{{i+1}}},{prefix}_{{{j+1}}})", font_size=27, color=prefix_color)
            row.add(VGroup(cell, text))
        row.arrange(RIGHT, buff=0.05)
        matrix.add(row)
    matrix.arrange(DOWN, buff=0.05)

    # row labels
    row_labels = VGroup(*[
        MathTex(f"{prefix}_{{{i+1}}}", font_size=26, color=color) for i in range(4)
    ])
    row_labels.arrange(DOWN, buff=0.8).next_to(matrix, LEFT, buff=0.5)

    # column labels
    col_labels = VGroup(*[
        MathTex(f"{prefix}_{{{i+1}}}", font_size=26, color=color) for i in range(4)
    ])
    col_labels.arrange(RIGHT, buff=1).next_to(matrix, UP, buff=0.5)

    matrix_group.add(matrix, row_labels, col_labels)
    matrix_group.move_to(position)
    return matrix_group





class Similarity_Pairwise_Explanation(Scene):
    def construct(self):
        title = make_title(self)
        step = Text("Step 4: Computing Pairwise Similarities", font_size=28).next_to(title, DOWN, buff=0.5)
        self.play(Write(step))
        expl = MathTex(
            r"\text{Cosine similarity is computed as: } \cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{||\mathbf{a}|| \, ||\mathbf{b}||}",
            font_size=30
        ).next_to(step, DOWN, buff=0.3)
        self.play(Write(expl))
    # now we have f = [f_1, f_2, f_3, f_4] and t = [t_1, t_2, t_3, t_4]
        f = ["f_1", "f_2", "f_3", "f_4"]
        t = ["t_1", "t_2", "t_3", "t_4"]
    # we want to compute all possible pairwise cosine similarities between f_i and f_j, and t_i and t_j for i != j
        expl = MathTex(f"{f, t},")
        
        # Show all combinations going through the cosine similarity function
        combinations = []
        # Add all f combinations
        for i in range(4):
            for j in range(i+1, 4):
                combinations.append((f"f_{i+1}", f"f_{j+1}", "f"))
        # Add all t combinations  
        for i in range(4):
            for j in range(i+1, 4):
                combinations.append((f"t_{i+1}", f"t_{j+1}", "t"))

        box = Rectangle(width=2.5, height=1, color=WHITE, fill_opacity=0.2)
        label = Text("cosine_similarity", font_size=20, color=YELLOW) # outputs cs(f_i, f_j) or cs(t_i, t_j)
        func = VGroup(box, label).move_to(ORIGIN + DOWN * 0.5)
        self.play(Create(func))

        # Calculate border positions based on function box center and width
        func_center = ORIGIN + DOWN * 0.5
        func_left_edge = func_center + LEFT * 1.5  # half width = 2.5/2 = 1.25
        func_right_edge = func_center + RIGHT * 2.0 

        first_iteration = True
        for vec1, vec2, vec_type in combinations:
            # Create input vectors from the left
            input1 = MathTex(vec1, font_size=35, color=GREEN if vec_type == "f" else RED).move_to(LEFT * 4)
            input2 = MathTex(vec2, font_size=35, color=GREEN if vec_type == "f" else RED).move_to(LEFT * 4 + DOWN * 0.5)
            self.play(Write(input1), Write(input2))
            
            # Move inputs to the left edge of the function box and fade them out
            # Make the first animation slower
            runtime = 1.0 if first_iteration else 0.2
            self.play(
                input1.animate.move_to(func_left_edge + UP * 0.25),
                input2.animate.move_to(func_left_edge + DOWN * 0.25),
                run_time=runtime
            )
            self.remove(input1, input2)
            
            # Show output appearing at the right edge of the function box
            output = MathTex(f"cs({vec1}, {vec2})", font_size=40, color=GREEN if vec_type == "f" else RED).move_to(func_right_edge)
            self.play(Write(output), run_time=0.2)
            
            # Move output to the right and fade out
            self.play(output.animate.move_to(RIGHT * 4),run_time=1.0 if first_iteration else 0.2)
            self.play(FadeOut(output), run_time=0.2)
            first_iteration = False

        self.wait(0.8)

In [ ]:
%%manim -qh Similarity_Matrices_Setup
class Similarity_Matrices_Setup(Scene):
    def construct(self):
        # title = make_title(self)
        # self.play(FadeOut(title))

        step = Text("Step 5: Building Similarity Matrices", font_size=28)
        self.play(Write(step))
        self.play(step.animate.to_edge(UP))

        m1 = create_empty_matrix_with_subscript_labels(LEFT * 3.5, "f", GREEN)
        m2 = create_empty_matrix_with_subscript_labels(RIGHT * 3.5, "t", RED)

        self.play(Create(m1), Create(m2))

        tt1 = MathTex(r"cs(f_i, f_j)", font_size=50, color=GREEN).next_to(m1, DOWN, buff=0.3)
        tt2 = MathTex(r"cs(t_i, t_j)", font_size=50, color=RED).next_to(m2, DOWN, buff=0.3)
        self.play(Write(tt1), Write(tt2))
        self.play(FadeOut(tt1), FadeOut(tt2))

        # Scale down and move matrices to bottom corners
        self.play(
            m1.animate.scale(0.7).move_to(LEFT * 4 + DOWN * 2),
            m2.animate.scale(0.7).move_to(RIGHT * 4 + DOWN * 2)
        )
        
        # Fade out the labels and title
        self.play(FadeOut(step))
        step = Text("Step 6: Comparing Similarity Matrices", font_size=28)  
        self.play(Write(step))
        self.play(step.animate.to_edge(UP))
        
        box = Rectangle(width=2.5, height=1, color=WHITE, fill_opacity=0.2)
        label = Text("cosine_similarity", font_size=20, color=YELLOW) # outputs cs(f_i, f_j) or cs(t_i, t_j)
        func = VGroup(box, label).move_to(ORIGIN + DOWN * 0.5)
        self.play(Create(func))
        
        # Animation for processing upper triangular values
        # Get the matrix components for easier access
        matrix_f = m1[0]  # The actual matrix cells
        matrix_t = m2[0]  # The actual matrix cells
        
        # Process each upper triangular coordinate
        for i, j in upper_tri_coords:
            # Get the cells from both matrices at position (i,j)
            # Each matrix[i][j] is a VGroup containing (cell, text)
            cell_f = matrix_f[i][j][0]  # Get the rectangle from the VGroup
            cell_t = matrix_t[i][j][0]  # Get the rectangle from the VGroup

            # Create copies of the similarity values to animate
            val_f = MathTex(f"cs(f_{{{i+1}}},f_{{{j+1}}})", font_size=24, color=GREEN)
            val_t = MathTex(f"cs(t_{{{i+1}}},t_{{{j+1}}})", font_size=24, color=RED)

            # Position the values at their matrix locations
            val_f.move_to(cell_f.get_center_of_mass())
            val_t.move_to(cell_t.get_center_of_mass())

            # Add the values to the scene
            self.add(val_f, val_t)
            
            # Move both values towards the function box (to the bottom)
            func_bottom = func.get_bottom()
            self.play(
                val_f.animate.move_to(func_bottom + LEFT * 0.6 + DOWN * 0.2),
                val_t.animate.move_to(func_bottom + RIGHT * 0.6 + DOWN * 0.2),
                run_time=0.8
            )
            
            # Brief pause as they "enter" the function
            # Remove the input values
            self.play(FadeOut(val_f), FadeOut(val_t),run_time=0.2)

            # Create the output value that emerges from the top
            output = MathTex(f"cs(cs(f_{{{i+1}}},f_{{{j+1}}}), cs(t_{{{i+1}}},t_{{{j+1}}}))", 
                           font_size=30, color=YELLOW)
            output.move_to(func.get_top() + UP * 0.2)
            
            # Show the output emerging and then fading
            self.play(Write(output), run_time=0.5)
            self.play(FadeOut(output), run_time=0.5)
            
            # Remove the original cells from the matrices (make them empty)
            # Replace with empty cells
            empty_cell_f = Rectangle(width=1.2, height=1.0, color=WHITE, fill_opacity=0.1, stroke_color=GRAY)
            empty_cell_t = Rectangle(width=1.2, height=1.0, color=WHITE, fill_opacity=0.1, stroke_color=GRAY)

            empty_cell_f.move_to(cell_f.get_center_of_mass())
            empty_cell_t.move_to(cell_t.get_center_of_mass())

            # Replace the cells
            self.play(
                Transform(cell_f, empty_cell_f),
                Transform(cell_t, empty_cell_t),
                run_time=0.3
            )
            
            self.wait(0.3)
        
        # Final message
        final_text = Text("All upper triangular similarities processed!", font_size=24, color=YELLOW)
        final_text.move_to(ORIGIN + UP * 1.5)
        self.play(Write(final_text))
        self.wait(1)